## 04_test_recommender.ipynb
### Test the fridge-to-recipe recommendation engine

In [16]:
# Cell 1: Imports
import polars as pl
import numpy as np
from sentence_transformers import SentenceTransformer
from rapidfuzz import fuzz
from pathlib import Path
import time

print("Libraries loaded.")

Libraries loaded.


In [17]:
# Cell 2: Load data
processed_path = Path('../processed')

df = pl.read_parquet(processed_path / "recipes_cleaned.parquet")
embeddings = np.load(processed_path / "recipe_embeddings.npy")

print(f"Loaded {df.shape[0]:,} recipes and embeddings of shape {embeddings.shape}")

Loaded 231,637 recipes and embeddings of shape (231637, 384)


In [18]:
# Cell 3: Load model (same one used for embeddings)
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6644.83it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.


In [19]:
# Cell 4: Targeted hybrid recommender (better for fridge use)
def recommend_recipes(user_ingredients: list[str], 
                     top_k: int = 10, 
                     allergy_list: list[str] = None,
                     min_coverage: float = 0.40):   # raised again
    
    if not user_ingredients:
        return []
    
    # Very focused query text — this helps a lot
    user_text = "Recipe using: " + ", ".join([ing.strip().lower() for ing in user_ingredients])
    user_emb = model.encode([user_text])[0]
    
    similarities = np.dot(embeddings, user_emb) / (
        np.linalg.norm(embeddings, axis=1) * np.linalg.norm(user_emb) + 1e-9
    )
    
    top_indices = np.argsort(similarities)[::-1][:top_k * 8]   # more candidates
    
    results = []
    for idx in top_indices:
        recipe = df.row(idx, named=True)
        recipe_ings = recipe.get("cleaned_ingredients", [])
        
        if len(recipe_ings) < 4 or len(recipe_ings) > 18:
            continue
            
        matches = 0
        missing = []
        for rec_ing in recipe_ings:
            # Use token_sort_ratio — better for ingredient names
            best_score = max((fuzz.token_sort_ratio(user_ing.lower(), rec_ing.lower()) 
                            for user_ing in user_ingredients), default=0)
            
            if best_score >= 78:          # stricter threshold
                matches += 1
            else:
                missing.append(rec_ing)
        
        coverage = matches / len(recipe_ings)
        
        if coverage < min_coverage:
            continue
        
        # Favor recipes that actually combine multiple user ingredients
        final_score = 0.50 * similarities[idx] + 0.50 * coverage
        
        # Allergy filter
        if allergy_list and any(
            any(fuzz.token_sort_ratio(allergen.lower(), ing.lower()) > 85 
                for ing in recipe_ings) 
            for allergen in allergy_list
        ):
            continue
        
        results.append({
            "name": recipe["name"],
            "score": round(final_score, 4),
            "coverage_pct": round(coverage * 100, 1),
            "missing_ingredients": missing[:5],
            "minutes": recipe.get("minutes", "N/A"),
            "id": recipe["id"]
        })
    
    results = sorted(results, key=lambda x: x["score"], reverse=True)[:top_k]
    return results

In [20]:
user_fridge = ["chicken", "rice", "broccoli", "soy sauce", "garlic"]

results = recommend_recipes(user_fridge, top_k=10, min_coverage=0.35)

print(f"\n=== Recommendations for: {user_fridge} ===\n")
for i, r in enumerate(results, 1):
    print(f"{i}. {r['name']}")
    print(f"   Score: {r['score']} | Coverage: {r['coverage_pct']}% | Time: {r['minutes']} min")
    if r.get('missing_ingredients'):
        print(f"   Missing: {', '.join(r['missing_ingredients'])}")
    print("-" * 80)


=== Recommendations for: ['chicken', 'rice', 'broccoli', 'soy sauce', 'garlic'] ===

1. erin s broccoli rice
   Score: 0.5810999870300293 | Coverage: 36.4% | Time: 35 min
   Missing: water, onion, sesame oil, olive oil, peanut butter
--------------------------------------------------------------------------------
2. chili garlic broccoli
   Score: 0.576200008392334 | Coverage: 40.0% | Time: 18 min
   Missing: red chili pepper, garlic clove, vegetable oil
--------------------------------------------------------------------------------
